# Lesson 1.3 — Coordinate frames and transformations

Every pose in the dataset is expressed **relative to something**. This notebook
makes that explicit, then verifies the relative vectors the policy receives.

Two traps are demonstrated rather than described:

1. the **quaternion component order** in `Pose.q`;
2. a **subtraction order error** that silently produces a plausible-looking but
   wrong relative vector.

## 1.3.1 — Preamble

In [1]:
import gymnasium as gym
import mani_skill.envs
import numpy as np
import torch

torch.set_printoptions(precision=4, sci_mode=False)


def make_env(obs_mode="state", control_mode="pd_joint_delta_pos", seed=0):
    env = gym.make(
        "PickCube-v1",
        obs_mode=obs_mode,
        control_mode=control_mode,
        num_envs=1,
    )
    env.reset(seed=seed)
    return env

## 1.3.2 — The `Pose` API: position and orientation

A 7-number pose is 3 translation values plus a 4-number quaternion. Which four,
and in what order, matters — get it wrong and the orientation is meaningless while
the translation still looks correct.

In [2]:
env = make_env()
agent = env.unwrapped.agent

pose = agent.tcp_pose
print("Pose object :", pose)
print("raw (7 values):", pose.raw_pose[0])
print("p (position)  :", pose.p[0])
print("q (quaternion):", pose.q[0])
print("\nThe same 7 values appear in the dataset as tcp_pose.")

2026-09-22 11:14:18,291 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1113: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 304: OS call failed or operation not supported on this OS (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  r = torch._C._cuda_getDeviceCount() if nvml_count < 0 else nvml_count


Pose object : Pose(raw_pose=tensor([[ 0.0123,  0.0380,  0.1822, -0.0177,  0.9998,  0.0043,  0.0080]]))
raw (7 values): tensor([ 0.0123,  0.0380,  0.1822, -0.0177,  0.9998,  0.0043,  0.0080])
p (position)  : tensor([0.0123, 0.0380, 0.1822])
q (quaternion): tensor([-0.0177,  0.9998,  0.0043,  0.0080])

The same 7 values appear in the dataset as tcp_pose.


## 1.3.3 — Trap: quaternion order in `Pose.q`

`Pose.q` is **(x, y, z, w)** — the scipy convention. SAPIEN documentation and many
robotics codebases use **(w, x, y, z)**. The two differ only by a rotation of the
components, so both "look like" a valid quaternion.

The test below converts the same numbers under both assumptions. With the TCP
pointing down, the correct interpretation yields a rotation that is close to a
clean single-axis rotation; the wrong one produces a large angle about an
unexpected axis.

> The magnitudes below were observed in a live session: the correct (x, y, z, w)
> reading gave approximately `[3.13, 0.016, -3.11]` radians, a near-pure rotation
> about z, while the (w, x, y, z) reading gave approximately
> `[-0.035, -0.016, 3.13]`.

In [3]:
from scipy.spatial.transform import Rotation as R

quat = pose.q[0].cpu().numpy()
print("raw quaternion values:", quat)

as_xyzw = R.from_quat(quat).as_euler("xyz")
as_wxyz = R.from_quat(np.roll(quat, 1)).as_euler("xyz")

print("\nif (x, y, z, w):", np.round(as_xyzw, 4), "rad ->", np.round(np.rad2deg(as_xyzw), 2), "deg")
print("if (w, x, y, z):", np.round(as_wxyz, 4), "rad ->", np.round(np.rad2deg(as_wxyz), 2), "deg")
print("\nPose.q is (x, y, z, w): the first reading is a near-pure z rotation.")

raw quaternion values: [-0.01768896  0.9998025   0.00428796  0.00798423]

if (x, y, z, w): [ 3.1333  0.0161 -3.1063] rad -> [ 179.52    0.92 -177.98] deg
if (w, x, y, z): [-0.0353 -0.0161  3.1333] rad -> [ -2.02  -0.92 179.52] deg

Pose.q is (x, y, z, w): the first reading is a near-pure z rotation.


## 1.3.4 — Homogeneous transforms and frame composition

A pose becomes a 4x4 homogeneous transform, which is what makes frame composition
and inversion mechanical: multiply to compose, invert to express something in the
opposite direction.

In [4]:
T = pose.to_transformation_matrix()[0].cpu().numpy()
np.set_printoptions(precision=4, suppress=True)
print("T (tcp -> world):")
print(T)

rotation = T[:3, :3]
print("\nR @ R.T == I (orthonormal):", np.allclose(rotation @ rotation.T, np.eye(3), atol=1e-5))
print("det(R) == 1 (right-handed, no reflection):", np.allclose(np.linalg.det(rotation), 1.0, atol=1e-5))

inverse = np.linalg.inv(T)
print("\ninv(T) @ T == I:", np.allclose(inverse @ T, np.eye(4), atol=1e-6))
print("Pose.inv().to_transformation_matrix() == np.linalg.inv(T):",
      np.allclose(pose.inv().to_transformation_matrix()[0].cpu().numpy(), inverse, atol=1e-5))

T (tcp -> world):
[[ 0.9998  0.0089  0.0158  0.0123]
 [ 0.0083 -0.9993  0.0354  0.038 ]
 [ 0.0161 -0.0353 -0.9992  0.1822]
 [ 0.      0.      0.      1.    ]]

R @ R.T == I (orthonormal): True
det(R) == 1 (right-handed, no reflection): True

inv(T) @ T == I: True
Pose.inv().to_transformation_matrix() == np.linalg.inv(T): True


## 1.3.5 — Frame composition with 4x4 matrices

A pose becomes a 4x4 homogeneous transform. That is what makes frame algebra
mechanical: **compose by multiplying, reverse by inverting**.

> Careful: `Pose.to(device)` moves a pose between devices. It is **not** a frame
> change. Frame composition is matrix algebra.

The cell below expresses the cube in the TCP frame and transforms it back:

In [5]:
def as_matrix(pose):
    """4x4 homogeneous transform of a Pose."""
    return pose.to_transformation_matrix()[0].cpu().numpy()


robot = env.unwrapped.agent.robot
tcp_pose = env.unwrapped.agent.tcp_pose
obj_pose = env.unwrapped.cube.pose

T_tcp = as_matrix(tcp_pose)
T_obj = as_matrix(obj_pose)

print("tcp  in world:\n", np.round(T_tcp, 4))
print("cube in world:\n", np.round(T_obj, 4))

# cube expressed in the TCP frame
T_obj_in_tcp = np.linalg.inv(T_tcp) @ T_obj
print("\ncube in TCP frame:\n", np.round(T_obj_in_tcp, 4))

# transform back to world
T_back = T_tcp @ T_obj_in_tcp
print("\nround trip == original cube pose:", np.allclose(T_back, T_obj, atol=1e-5))

print("\nPose.inv() agrees with matrix inversion:",
      np.allclose(as_matrix(tcp_pose.inv()), np.linalg.inv(T_tcp), atol=1e-5))

tcp  in world:
 [[ 0.9998  0.0089  0.0158  0.0123]
 [ 0.0083 -0.9993  0.0354  0.038 ]
 [ 0.0161 -0.0353 -0.9992  0.1822]
 [ 0.      0.      0.      1.    ]]
cube in world:
 [[-0.353  -0.9356  0.     -0.0007]
 [ 0.9356 -0.353   0.      0.0536]
 [ 0.      0.      1.      0.02  ]
 [ 0.      0.      0.      1.    ]]

cube in TCP frame:
 [[-0.3452 -0.9384  0.0161 -0.0155]
 [-0.9381  0.3445 -0.0353 -0.01  ]
 [ 0.0276 -0.0273 -0.9992  0.1624]
 [ 0.      0.      0.      1.    ]]

round trip == original cube pose: True

Pose.inv() agrees with matrix inversion: True


## 1.3.6 — Predict the relative vectors, then verify

The observation contains `tcp_to_obj_pos` and `obj_to_goal_pos`. Before running the
next cell, decide what they are. Two plausible readings:

- **A**: a subtraction of two world-frame positions, `obj_pos - tcp_pos`;
- **B**: the object's position *expressed in the TCP frame*, i.e. rotated into that
  frame.

Both are useful, both are shape `(3,)`, and they are **numerically different** unless
the TCP orientation is identity. Only one of them is in the observation.

In [6]:
obs = env.unwrapped.get_obs().clone()[0]

tcp_pose_obs = obs[19:26]
goal_pos = obs[26:29]
obj_pose_obs = obs[29:36]
tcp_to_obj = obs[36:39]
obj_to_goal = obs[39:42]

print("tcp_to_obj_pos == obj_pos - tcp_pos  (world-frame difference):",
      np.allclose(tcp_to_obj.cpu(), (obj_pose_obs[:3] - tcp_pose_obs[:3]).cpu(), atol=1e-5))
print("obj_to_goal_pos == goal_pos - obj_pos (world-frame difference):",
      np.allclose(obj_to_goal.cpu(), (goal_pos - obj_pose_obs[:3]).cpu(), atol=1e-5))

print("\nthe opposite subtraction orders (also plausible, also wrong):")
print("  tcp_to_obj_pos == tcp_pos - obj_pos:",
      np.allclose(tcp_to_obj.cpu(), (tcp_pose_obs[:3] - obj_pose_obs[:3]).cpu(), atol=1e-5))
print("  obj_to_goal_pos == obj_pos - goal_pos:",
      np.allclose(obj_to_goal.cpu(), (obj_pose_obs[:3] - goal_pos).cpu(), atol=1e-5))

print("\nactual values: tcp_to_obj =", np.round(tcp_to_obj.cpu().numpy(), 6),
      " obj_to_goal =", np.round(obj_to_goal.cpu().numpy(), 6))

tcp_to_obj_pos == obj_pos - tcp_pos  (world-frame difference): True
obj_to_goal_pos == goal_pos - obj_pos (world-frame difference): True

the opposite subtraction orders (also plausible, also wrong):
  tcp_to_obj_pos == tcp_pos - obj_pos: False
  obj_to_goal_pos == obj_pos - goal_pos: False

actual values: tcp_to_obj = [-0.013   0.0156 -0.1622]  obj_to_goal = [ 0.0276 -0.0556  0.2689]


### Reading A is the correct one — and reading B gives a different vector

`tcp_to_obj_pos` is `obj_pos - tcp_pos`. Both are world-frame positions, so the
result is a **world-frame displacement**, not a rotated one.

To make the distinction concrete, compute reading B from the same observation. The
TCP is pointing downward, so its rotation is not identity, and the two results differ
in every component:

In [7]:
from mani_skill.utils.structs.pose import Pose

# Rebuild Pose objects from the observed 7-value poses
tcp_obs = Pose.create_from_pq(p=tcp_pose_obs[:3], q=tcp_pose_obs[3:7])
obj_obs = Pose.create_from_pq(p=obj_pose_obs[:3], q=obj_pose_obs[3:7])

T_tcp_obs = as_matrix(tcp_obs)
T_obj_obs = as_matrix(obj_obs)

R_rel = np.linalg.inv(T_tcp_obs)[:3, :3]
world_diff = T_obj_obs[:3, 3] - T_tcp_obs[:3, 3]
obj_in_tcp_frame = R_rel @ world_diff

print("R (TCP->world) rotation block:\n", np.round(R_rel, 4))
print("is the rotation close to identity?", np.allclose(R_rel, np.eye(3), atol=1e-3))
print()
print("A: world-frame difference  obj_pos - tcp_pos :", np.round(world_diff, 6))
print("B: object in TCP frame     R^-1 @ difference :", np.round(obj_in_tcp_frame, 6))
print("observed tcp_to_obj_pos                      :", np.round(tcp_to_obj.cpu().numpy(), 6))
print()
print("A matches the observation:", np.allclose(tcp_to_obj.cpu(), world_diff, atol=1e-5))
print("B matches the observation:", np.allclose(tcp_to_obj.cpu(), obj_in_tcp_frame, atol=1e-5))

R (TCP->world) rotation block:
 [[ 0.9998  0.0083  0.0161]
 [ 0.0089 -0.9993 -0.0353]
 [ 0.0158  0.0354 -0.9992]]
is the rotation close to identity? False

A: world-frame difference  obj_pos - tcp_pos : [-0.013   0.0156 -0.1622]
B: object in TCP frame     R^-1 @ difference : [-0.0155 -0.01    0.1624]
observed tcp_to_obj_pos                      : [-0.013   0.0156 -0.1622]

A matches the observation: True
B matches the observation: False


### Why this distinction matters

The observation gives you a world-frame displacement vector. If you need the object
in the TCP frame — to plan a motion relative to the gripper, for example — you must
rotate it yourself:

```text
obj_in_tcp = R_tcp^{-1} @ (obj_pos - tcp_pos)
```

Using `tcp_to_obj_pos` directly as if it were already in the TCP frame is a silent
frame error: every number is plausible, and the resulting motion is wrong.

## 1.3.7 — Why frame conventions block dataset merging

Before combining two datasets, at least these must agree or be transformed:

1. **handedness and axis convention** — a rotation that is correct in one
   convention is wrong in another;
2. **quaternion component order** — `xyzw` versus `wxyz`;
3. **which frame** each pose is expressed in — world, robot base, or camera;
4. **the sign of relative vectors** — as demonstrated above.

None of these change a tensor's shape. A pair of datasets can be shape-compatible
and still represent the same physical motion with opposite sign.

This is why Lesson 4 treats rotation representations and frame composition as its
own subject, and why `docs/roadmap_v3.md` asks the same question for sim/VR/real
data.

## Takeaways

1. `Pose.q` is `(x, y, z, w)`. Verify orientation conventions empirically; never
   assume wxyz.
2. Frame composition is 4x4 matrix algebra: multiply to compose, invert to reverse.
   `Pose.to()` is a device move, not a frame change.
3. `tcp_to_obj_pos` and `obj_to_goal_pos` are **world-frame position differences**,
   not vectors rotated into the gripper frame. Computing the TCP-frame version
   requires `R_tcp^{-1}` and yields different numbers.
4. Two shape-compatible datasets can still be mutually unusable, because none of
   handedness, quaternion order, reference frame, or vector sign affects shape.